# 05 — Ranked incidents and locked holdout

Development incidents are available immediately. Set `RUN_HOLDOUT=1`
only after the selected configuration is accepted and frozen. Holdout can
be opened once; a receipt prevents accidental reuse.


## 1. Setup


In [ ]:
import importlib
import hashlib
import os
import sys
import tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT") or os.getenv("ANOMALY_DRIVE_ROOT")
    or ("/content/drive/MyDrive/anomaly_detection" if IN_COLAB
        else Path.home() / "anomaly_detection_data")
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

SECTOR = os.getenv("ANOMALY_SECTOR", "telecom")
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_11_run1",
    "petrobras_3w": "petrobras_3w_core_v0_11_run1",
}
from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json
import evaluation_core, simple_model_core
evaluation_core = importlib.reload(evaluation_core)
simple_model_core = importlib.reload(simple_model_core)
from evaluation_core import ALERT_COLUMNS, evaluate_cases, form_cases
from simple_model_core import (
    alerts_from_score_file, case_score_trace, fit_topology_reference,
    materialize_measurement_features, materialize_wide_partition,
    merge_score_files, partition_exposure, score_residual_file,
    score_topology_file,
)

RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / os.getenv(
    "CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR]
)
CORE_ROOT, SPLIT_ROOT = RUN_ROOT / "SPEC-CORE", RUN_ROOT / "SPLITS"
VERSION = "3.0.0"
MODEL_ROOT = DATA_ROOT / "outputs" / "models" / f"v{VERSION}" / SECTOR / f"{SECTOR}_models_v3_run1"
FINAL_ROOT = DATA_ROOT / "outputs" / "cases" / f"v{VERSION}" / SECTOR / f"{SECTOR}_cases_v3_run1"
EVALUATION_ROOT = DATA_ROOT / "outputs" / "evaluation" / f"v{VERSION}" / SECTOR / f"{SECTOR}_evaluation_v3_run1"
RUN_HOLDOUT = os.getenv("RUN_HOLDOUT", "0") == "1"

configuration = read_json(MODEL_ROOT / "selected_configuration.json")
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
bundle = joblib.load(MODEL_ROOT / "residual_bundle.joblib")
topology_path = CORE_ROOT / "topology_memberships.parquet"
topology = pd.read_parquet(topology_path) if topology_path.is_file() else pd.DataFrame()
partition = "holdout" if RUN_HOLDOUT else "development"
if RUN_HOLDOUT and configuration["selection_status"] != "within_budget":
    raise RuntimeError("Holdout remains sealed because development missed the workload gate")
display(pd.Series({
    "partition": partition, "portfolio": configuration["portfolio"],
    "topology": not topology.empty, "output": FINAL_ROOT,
}, name="value").to_frame())


## 2. Load development or score the one-time holdout


In [ ]:
if not RUN_HOLDOUT:
    alerts = pd.read_parquet(MODEL_ROOT / "selected_alerts.parquet")
    cases = pd.read_parquet(MODEL_ROOT / "selected_cases.parquet")
    members = pd.read_parquet(MODEL_ROOT / "selected_case_members.parquet")
    metrics = pd.read_csv(MODEL_ROOT / "development_metrics.csv")
    fault_types = pd.read_csv(MODEL_ROOT / "fault_type_results.csv")
    domain_types = pd.read_csv(MODEL_ROOT / "domain_type_results.csv")
    localisation = pd.read_csv(MODEL_ROOT / "localisation_results.csv")
    trace = pd.read_parquet(MODEL_ROOT / "development_case_trace.parquet")
else:
    receipt = FINAL_ROOT.parent / "HOLDOUT_USED.json"
    if receipt.exists():
        raise RuntimeError(f"Holdout already used: {receipt}")
    truth_root = EVALUATION_ROOT / "holdout_sealed"
    fault_events = pd.read_parquet(truth_root / "fault_events.parquet")
    fault_intervals = pd.read_parquet(truth_root / "fault_entity_intervals.parquet")
    settings = configuration["feature_settings"]
    horizon = configuration.get("decision_horizon_seconds_by_fault_type") or configuration["decision_horizon_seconds"]

    temporary = tempfile.TemporaryDirectory()
    work = Path(temporary.name)
    wide, features = work / "wide.parquet", work / "features.parquet"
    self_scores, residuals = work / "self_scores.parquet", work / "residuals.parquet"
    split = materialize_wide_partition(
        CORE_ROOT, SPLIT_ROOT, "holdout", catalogue, wide,
        lookback_seconds=configuration["lookback_seconds"],
    )
    materialize_measurement_features(
        wide, catalogue, features,
        target_cadence_seconds=settings["base_cadence_seconds"],
    )
    score_residual_file(
        bundle, features, self_scores,
        cadence_seconds=settings["base_cadence_seconds"],
        dispersion_window_seconds=settings["dispersion_window_seconds"],
        cusum_allowance=configuration["cusum_allowance"],
        residual_destination=residuals,
        score_start=split["score_start"], score_end=split["score_end"],
    )
    scores = self_scores
    if configuration["topology_enabled"]:
        reference = pd.read_csv(MODEL_ROOT / "topology_reference.csv")
        decisions = configuration["topology_decisions"]
        topology_scores, combined = work / "topology_scores.parquet", work / "scores.parquet"
        score_topology_file(
            residuals, topology, reference, topology_scores,
            peer_group_type=decisions["primary_peer_level"],
            group_types=decisions["common_mode_levels"],
            min_peers=decisions["minimum_valid_peers"],
            min_group_entities=decisions["minimum_group_entities"],
            min_group_fraction=decisions["minimum_group_available_fraction"],
        )
        merge_score_files(self_scores, topology_scores, combined)
        scores = combined

    frames = [
        alerts_from_score_file(
            scores, channel, configuration["thresholds"][channel],
            min_consecutive=configuration["channel_persistence_observations"][channel],
            recovery_consecutive=configuration["recovery_observations"],
        ) for channel in configuration["channels"]
    ]
    frames = [frame for frame in frames if not frame.empty]
    alerts = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=ALERT_COLUMNS)
    if not alerts.empty:
        alerts = alerts.sort_values("alert_start").reset_index(drop=True)
        alerts["alert_id"] = [f"A-{number:09d}" for number in range(1, len(alerts) + 1)]
    cases, members = form_cases(
        alerts, topology, gap_seconds=configuration["case_gap_seconds"],
        thresholds=configuration["thresholds"],
    )
    exposure = partition_exposure(
        scores, configuration["exposure_unit"], settings["base_cadence_seconds"]
    )
    result = evaluate_cases(
        cases, members, fault_events, fault_intervals,
        exposure_value=exposure, exposure_unit=configuration["exposure_unit"],
        decision_horizon_seconds=horizon, topology_memberships=topology,
    )
    metrics, fault_types = result["metrics"], result["fault_type_results"]
    domain_types = result["domain_type_results"]
    localisation = result["localisation_results"]
    trace = case_score_trace(
        scores, features, cases, members, alerts, bundle,
        configuration["thresholds"], configuration["channels"],
        window_seconds=max(settings["dispersion_window_seconds"], 20 * settings["base_cadence_seconds"]),
    )


## 3. Rank and save traceable incidents


In [ ]:
cases = cases.sort_values(
    ["anomaly_evidence_score", "case_start"], ascending=[False, True]
).reset_index(drop=True)
if "rank" in cases:
    cases = cases.drop(columns="rank")
cases.insert(0, "rank", np.arange(1, len(cases) + 1))
cases["operational_priority"] = pd.NA
cases["priority_status"] = "not_scored_no_validated_impact_data"
display(cases.head(25))
display(metrics)
display(fault_types)
display(domain_types)

with new_output_directory(FINAL_ROOT) as output:
    cases.to_csv(output / "ranked_incidents.csv", index=False)
    alerts.to_parquet(output / "alerts.parquet", index=False)
    members.to_parquet(output / "incident_members.parquet", index=False)
    trace.to_parquet(output / "incident_score_trace.parquet", index=False)
    localisation.to_csv(output / "localisation_results.csv", index=False)
    metrics.to_csv(output / "evaluation_metrics.csv", index=False)
    fault_types.to_csv(output / "fault_type_results.csv", index=False)
    domain_types.to_csv(output / "domain_type_results.csv", index=False)
    write_json(output / "incident_run.json", {
        "version": VERSION, "sector": SECTOR, "partition": partition,
        "configuration_sha256": hashlib.sha256(
            (MODEL_ROOT / "selected_configuration.json").read_bytes()
        ).hexdigest(),
        "ranking": "normalised anomaly evidence only",
        "localisation": "probable observable scope, not proven root cause",
        "synthetic_limit": configuration["synthetic_limit"],
    })
if RUN_HOLDOUT:
    write_json(FINAL_ROOT.parent / "HOLDOUT_USED.json", {
        "sector": SECTOR, "configuration": str(MODEL_ROOT), "result": str(FINAL_ROOT)
    })
    temporary.cleanup()
print("Saved:", FINAL_ROOT)
print("Next: 06_PRODUCT_DEMO.ipynb")
